In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/__results__.html
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/__huggingface_repos__.json
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/__notebook__.ipynb
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/reviews_with_sentiment.csv
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/__output__.json
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/custom.css
/kaggle/input/datasets/kylin0338/pairswithprobs/pairs_with_probs.csv
/kaggle/input/datasets/kylin0338/error-finding/sentiment_error_analysis.csv.xlsx
/kaggle/input/datasets/kylin0338/updated-validation/validation.csv.xlsx
/kaggle/input/datasets/charitarth/semeval-2014-task-4-aspectbasedsentimentanalysis/Laptop_Train_v2.xml
/kaggle/input/datasets/charitarth/semeval-2014-task-4-aspectbasedsentimentanalysis/Restaurants_Test_Data_PhaseB.csv
/kaggle/input/datasets/charitarth/semeval-2014-task-4-aspectbasedsentimentanalysis/laptops-trial.csv
/kagg

In [2]:
import pandas as pd, spacy
from collections import Counter

df = pd.read_csv ("/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/reviews_with_sentiment.csv")
df["review_id"] = df.index
nlp = spacy.load("en_core_web_sm", disable=["ner"])

In [3]:
sample = df["text"].sample(5000, random_state=1).tolist()
counter = Counter()
for doc in nlp.pipe(sample, batch_size=256):
    for chunk in doc.noun_chunks:
        counter[chunk.root.lemma_.lower()] += 1

print(counter.most_common(100))

[('i', 11853), ('it', 8457), ('phone', 7867), ('you', 2179), ('that', 1562), ('this', 1371), ('they', 1194), ('screen', 973), ('camera', 687), ('app', 675), ('battery', 662), ('issue', 578), ('which', 549), ('time', 547), ('price', 540), ('life', 518), ('problem', 492), ('what', 471), ('card', 459), ('one', 414), ('thing', 407), ('quality', 396), ('device', 382), ('day', 374), ('product', 369), ('we', 361), ('feature', 344), ('iphone', 336), ('case', 328), ('everything', 327), ('call', 324), ('charger', 301), ('month', 295), ('samsung', 289), ('review', 274), ('note', 272), ('he', 259), ('picture', 253), ('all', 253), ('money', 250), ('amazon', 246), ('service', 245), ('update', 241), ('condition', 239), ('use', 234), ('seller', 222), ('lot', 219), ('year', 211), ('box', 207), ('verizon', 194), ('work', 193), ('photo', 193), ('button', 192), ('speaker', 190), ('hour', 187), ('people', 183), ('version', 183), ('video', 181), ('scratch', 175), ('android', 172), ('way', 171), ('edge', 168

In [4]:
ASPECTS = {
    "battery": ["battery", "battery life", "charge", "charger", "charging", "drain", "power bank"],
    "camera": ["camera", "photo", "picture", "video", "selfie", "lens", "flash", "autofocus"],
    "screen": ["screen", "display", "touchscreen", "brightness", "resolution", "amoled", "lcd"],
    "price": ["price", "cost", "value for money", "worth", "affordable", "cheap", "expensive", "overpriced"],
    "performance": ["performance", "speed", "lag", "laggy", "freeze", "crash", "processor", "ram", "smooth", "multitask"],
    "sound": ["sound", "speaker", "volume", "audio", "ringtone", "earpiece", "headphone jack"],
    "build": ["build quality", "durable", "durability", "material", "fragile", "sturdy", "scratch", "cracked", "plastic", "metal body"],
    "design": ["design", "size", "weight", "lightweight", "color", "slim", "thin", "sleek", "looks", "bezel"],
    "usability": ["easy to use", "user friendly", "interface", "intuitive", "fingerprint", "face unlock", "navigation"],
    "network": ["network", "signal", "reception", "carrier", "verizon", "sim", "lte", "4g", "5g", "wifi", "dropped calls", "coverage"],
    "storage": ["storage", "sd card", "microsd", "memory", "internal storage"],
    "delivery": ["shipping", "delivery", "delivered", "arrived", "package", "packaging", "box"],
    "customer_service": ["seller", "customer service", "support", "refund", "return", "warranty", "replacement"],
    "software" : ["software", "operating system", "firmware", "bloatware", "update", "updates", "upgrade", "android version", "ios", "pre-installed", "ui"],
}

In [5]:
# import re

# PATTERNS = {
#     asp: re.compile(
#         r"\b(?:" + "|".join(re.escape(w) for w in words) + r")s?\b"
#     )
#     for asp, words in ASPECTS.items()
# }

# def find_aspects(sentence):
#     s = sentence.lower()
#     return [
#         asp for asp, pat in PATTERNS.items()
#         if pat.search(s)
#     ]
# texts = df["text"].tolist()
# ids = df["review_id"].tolist()
# rows = []
# for rid, doc in zip(ids, nlp.pipe(texts, batch_size=256)):
#     for sent in doc.sents:
#         found = find_aspects(sent.text)
#         if 1 <= len(found) <= 2 and len(sent.text.split()) >= 4:
#             for asp in found:
#                 rows.append({"review_id": rid, "sentence": sent.text.strip(), "aspect": asp})

In [6]:
import re
# in ASPECTS["software"]: replace "upgrade" with specific phrases
ASPECTS["software"] = ["software", "operating system", "firmware", "bloatware", "update", "updates",
                       "software upgrade", "system upgrade", "android upgrade", "ios", "pre-installed", "ui"]
EXCLUDE = re.compile(
    r"\b(home screen|lock screen|screen protector|screen cover|phone case|opened the box"
    r"|email updates?|status updates?|updates? (?:about|on) (?:the )?(?:status|order|shipment))\b"
    r"|^\W*(?:edit|update)\s*:"
)

PATTERNS = {
    asp: re.compile(r"\b(?:" + "|".join(re.escape(w) for w in words) + r")s?\b")
    for asp, words in ASPECTS.items()
}
def find_aspects(sentence):
    s = EXCLUDE.sub(" ", sentence.lower())
    return [asp for asp, pat in PATTERNS.items() if pat.search(s)]

# 4) same extraction loop as before, unchanged
texts = df["text"].tolist()
ids = df["review_id"].tolist()
rows = []
for rid, doc in zip(ids, nlp.pipe(texts, batch_size=256)):
    for sent in doc.sents:
        found = find_aspects(sent.text)
        if 1 <= len(found) <= 2 and len(sent.text.split()) >= 4:
            for asp in found:
                rows.append({"review_id": rid, "sentence": sent.text.strip(), "aspect": asp})

pairs = pd.DataFrame(rows)
print(pairs.shape)
print(pairs["aspect"].value_counts(normalize=True).round(3))

(43230, 3)
aspect
battery             0.157
network             0.122
camera              0.114
screen              0.110
price               0.084
customer_service    0.081
design              0.060
software            0.057
delivery            0.047
sound               0.042
performance         0.041
usability           0.032
storage             0.028
build               0.025
Name: proportion, dtype: float64


In [7]:
import numpy as np

def make_key(d):
    return d["review_id"].astype(str) + "|" + d["aspect"] + "|" + d["sentence"]

pairs["key"] = make_key(pairs)

old = pd.read_csv("/kaggle/input/datasets/kylin0338/pairswithprobs/pairs_with_probs.csv")
old["key"] = make_key(old)
old = old[["key", "p_neg", "p_neu", "p_pos"]].drop_duplicates("key")

pairs = pairs.merge(old, on="key", how="left")
missing = pairs["p_neg"].isna()
print(missing.sum(), "new pairs to score;", (~missing).sum(), "reused")

870 new pairs to score; 42360 reused


In [8]:
from transformers import pipeline

absa = pipeline("text-classification", model="yangheng/deberta-v3-base-absa-v1.1",
                device=0, truncation=True, max_length=256)

def probs(o):
    d = {x["label"].lower(): x["score"] for x in o}
    return d.get("negative", 0), d.get("neutral", 0), d.get("positive", 0)

new = pairs[missing]
inputs = [{"text": s, "text_pair": a} for s, a in zip(new["sentence"], new["aspect"])]
out = absa(inputs, batch_size=64, top_k=None)
pairs.loc[missing, ["p_neg", "p_neu", "p_pos"]] = np.array([probs(o) for o in out])

labs = np.array(["negative", "neutral", "positive"])
P = pairs[["p_neg", "p_neu", "p_pos"]].values
pairs["aspect_sentiment"] = labs[P.argmax(axis=1)]
pairs["aspect_score"] = P.max(axis=1)

pairs = pairs.merge(df[["review_id", "asin", "rating"]], on="review_id", how="left")
pairs.to_csv("/kaggle/working/pairs_with_probs_v2.csv", index=False)
print(pairs.shape, pairs["aspect_sentiment"].value_counts(normalize=True).round(3))

config.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/372 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

(43230, 11) aspect_sentiment
positive    0.406
negative    0.371
neutral     0.223
Name: proportion, dtype: float64


In [9]:
# sanity check on the new probabilities
print((pairs[["p_neg", "p_neu", "p_pos"]].sum(axis=1)).describe().round(3))   # should be ~1.0
print(pairs[["p_neg", "p_neu", "p_pos"]].isna().sum())                        # should be 0



count    43230.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
dtype: float64
p_neg    0
p_neu    0
p_pos    0
dtype: int64


In [10]:
pairs = pd.DataFrame(rows)

print("Aspect distribution:")
print(pairs["aspect"].value_counts(normalize=True).round(3))

print("\nSample rows:")
print(pairs.sample(min(10, len(pairs)), random_state=42).to_string(index=False))
print(
    pairs.sample(min(10, len(pairs)), random_state=42)
         .to_string(index=False)
)
# print(pd.crosstab(pairs["aspect"], pairs["aspect_sentiment"], normalize="index").round(2))

Aspect distribution:
aspect
battery             0.157
network             0.122
camera              0.114
screen              0.110
price               0.084
customer_service    0.081
design              0.060
software            0.057
delivery            0.047
sound               0.042
performance         0.041
usability           0.032
storage             0.028
build               0.025
Name: proportion, dtype: float64

Sample rows:
 review_id                                                                                                                                                                                                                                                                                         sentence      aspect
      6447                                                                                                                                                                  E.g. the flimsy plastic back shell is so bad that a whole third party busines

In [11]:
pairs = pd.DataFrame(rows)
print(pairs.shape)
print(pairs["aspect"].value_counts(normalize=True).round(3))
# print(pd.crosstab(pairs["aspect"], pairs["aspect_sentiment"], normalize="index").round(2))

(43230, 3)
aspect
battery             0.157
network             0.122
camera              0.114
screen              0.110
price               0.084
customer_service    0.081
design              0.060
software            0.057
delivery            0.047
sound               0.042
performance         0.041
usability           0.032
storage             0.028
build               0.025
Name: proportion, dtype: float64


In [12]:
from transformers import pipeline

absa = pipeline(
    "text-classification",
    model="yangheng/deberta-v3-base-absa-v1.1",
    device=0, truncation=True, max_length=256
)

inputs = [{"text": s, "text_pair": a} for s, a in zip(pairs["sentence"], pairs["aspect"])]
out = absa(inputs, batch_size=64)

pairs["aspect_sentiment"] = [o["label"].lower() for o in out]
pairs["aspect_score"] = [o["score"] for o in out]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [13]:
print(pd.crosstab(pairs["aspect"], pairs["aspect_sentiment"], normalize="index").round(2))

aspect_sentiment  negative  neutral  positive
aspect                                       
battery               0.40     0.25      0.35
build                 0.36     0.14      0.50
camera                0.26     0.29      0.45
customer_service      0.54     0.27      0.18
delivery              0.29     0.17      0.54
design                0.20     0.08      0.72
network               0.45     0.35      0.21
performance           0.31     0.08      0.61
price                 0.25     0.10      0.64
screen                0.43     0.18      0.39
software              0.40     0.36      0.25
sound                 0.50     0.13      0.36
storage               0.34     0.35      0.31
usability             0.33     0.09      0.58


In [14]:
pairs = pairs.merge(df[["review_id", "asin", "rating"]], on="review_id")
pairs.to_csv("/kaggle/working/aspect_sentiment.csv", index=False)

In [15]:
for asp in ["design", "usability", "performance", "sound"]:
    # print(f"\n=== {asp} ===")
    for _, r in pairs[pairs["aspect"] == asp].sample(12, random_state=3).iterrows():
        print(f"[{r['aspect_sentiment']}] {r['sentence'][:150]}")

[negative] Many times, the design of the phone's OS forces the phone to work way harder than is really necessary to achieve its tasks.
[positive] The screen looks great, and apps generally run well.
[negative] but it's very slim and definitely pocket-sized.
[positive] Movies fill the entire screen and the clarity/color is terrific.
[positive] The phone looks awesome and I verified an local SIM and it works fine.
[neutral] You can change text size, set individual ringtones and notification sounds, change the shortcuts and the functions of the side buttons.
[positive] Phone looks new as advertised n works great.
[positive] The phone is beautiful, sleek, & powerful... and unfortunately a complete & total lemon.
[negative] Also, the Convoy 3 is quite a bit smaller than the Convoy 4, with no advantage in screen size.
[neutral] It used a long exposure, and the resulting photo looks like I took it on a normal day.
[positive] Great phone looks amazing just the camera still isn't the very best 

In [16]:
val = pairs.groupby("aspect").sample(20, random_state=1)[
    ["review_id", "sentence", "aspect", "aspect_sentiment", "aspect_score"]
]
val["aspect_correct"] = ""      # 1 if sentence is really about that aspect, else 0
val["sentiment_correct"] = ""   # 1 if the sentiment label is right, else 0
val.to_csv("/kaggle/working/validation.csv", index=False)

In [17]:
import os
print(os.path.exists(
    "/kaggle/working/validation.csv"))

True


In [18]:
pairs.to_csv(
    "/kaggle/working/aspect_sentiment.csv",
    index=False
)

In [19]:
import os
print(os.path.exists(
    "/kaggle/working/aspect_sentiment.csv"))

True


In [20]:
import pandas as pd

val = pd.read_excel("/kaggle/input/datasets/kylin0338/updated-validation/validation.csv.xlsx")

print(val.head())
print(val.shape)
print(val.columns.tolist())

   review_id                                           sentence   aspect  \
0       6719  this phone does not hold a good battery charge...  battery   
1      15636  Only Con was that it didn't come with a adapte...  battery   
2       7156  However, now, I charge it every other night wi...  battery   
3       7638  Battery life is improved but no cigar compared...  battery   
4      17304       Within 90 minutes it was down to 95% charge.  battery   

  aspect_sentiment  aspect_score  aspect_correct  sentiment_correct  
0         negative      0.990458               1                  1  
1         negative      0.764530               0                  1  
2          neutral      0.610037               1                  0  
3         positive      0.995783               1                  0  
4         negative      0.936439               1                  0  
(280, 7)
['review_id', 'sentence', 'aspect', 'aspect_sentiment', 'aspect_score', 'aspect_correct', 'sentiment_correct']


In [21]:
print(val["aspect_correct"].value_counts(dropna=False))
print(val["sentiment_correct"].value_counts(dropna=False))

aspect_correct
1    253
0     27
Name: count, dtype: int64
sentiment_correct
1    214
0     66
Name: count, dtype: int64


In [22]:
# Aspect detection precision
aspect_precision = (
    val.groupby("aspect")["aspect_correct"].mean() * 100
)

print(aspect_precision.round(2))


aspect
battery              90.0
build               100.0
camera               95.0
customer_service     90.0
delivery             85.0
design               80.0
network             100.0
performance          75.0
price                95.0
screen               85.0
software             80.0
sound               100.0
storage              95.0
usability            95.0
Name: aspect_correct, dtype: float64


In [23]:
# Sentiment accuracy for correctly detected aspects
correct = val[val["aspect_correct"] == 1]

sentiment_accuracy = (
    correct.groupby("aspect")["sentiment_correct"].mean() * 100
)

print(sentiment_accuracy.round(2))

aspect
battery             72.22
build               75.00
camera              78.95
customer_service    83.33
delivery            94.12
design              81.25
network             55.00
performance         80.00
price               68.42
screen              88.24
software            75.00
sound               75.00
storage             89.47
usability           84.21
Name: sentiment_correct, dtype: float64


In [24]:
# Incorrect aspect predictions
aspect_errors = val[val["aspect_correct"] == 0]

# Incorrect sentiment predictions
sentiment_errors = val[
    (val["aspect_correct"] == 1) &
    (val["sentiment_correct"] == 0)
]

print("Aspect errors:", len(aspect_errors))
print("Sentiment errors:", len(sentiment_errors))

display(sentiment_errors.head(15))

Aspect errors: 27
Sentiment errors: 55


,review_id,sentence,aspect,aspect_sentiment,aspect_score,aspect_correct,sentiment_correct
2,7156,"However, now, I charge it every other night wi...",battery,neutral,0.610037,1,0
3,7638,Battery life is improved but no cigar compared...,battery,positive,0.995783,1,0
4,17304,Within 90 minutes it was down to 95% charge.,battery,negative,0.936439,1,0
11,3657,"-Battery life takes some time to ""even out"".",battery,negative,0.991018,1,0
17,3821,I have to dock the battery life a star because...,battery,neutral,0.696890,1,0
20,16208,"Thereâ€™s no big scratch, charging port is lit...",build,positive,0.513337,1,0
23,10376,Press down on the home button and the screen c...,build,negative,0.736291,1,0
28,5415,People have complained about the rear mounted ...,build,negative,0.405939,1,0
32,3096,"The phone is not fragile, many nights I lay on...",build,neutral,0.955093,1,0
33,8067,Build Quality - Not too bad but the Asus is wa...,build,neutral,0.730827,1,0


In [25]:
pd.set_option("display.max_colwidth", None)

display(
    sentiment_errors[
        ["sentence", "aspect", "aspect_sentiment", "aspect_score"]
    ].to_string(index=False)
)

'                                                                                                                                                                                                                                                                                              sentence           aspect aspect_sentiment  aspect_score\n                                                                                                                                                                                                                                          However, now, I charge it every other night with normal use.          battery          neutral      0.610037\n                                                                                                                                                                                                                                             Battery life is improved but no cigar compared to iPhone.          battery

In [26]:
print(
    sentiment_errors["aspect"]
    .value_counts()
)

aspect
network             9
price               6
battery             5
sound               5
build               5
camera              4
software            4
usability           3
customer_service    3
design              3
performance         3
screen              2
storage             2
delivery            1
Name: count, dtype: int64


In [27]:
print(
    sentiment_errors["aspect_sentiment"]
    .value_counts()
)

aspect_sentiment
neutral     27
negative    17
positive    11
Name: count, dtype: int64


In [28]:
# Extract incorrect sentiment predictions
sentiment_errors = val[
    (val["aspect_correct"] == 1) &
    (val["sentiment_correct"] == 0)
].copy()

# Add a column for manually verified sentiment
sentiment_errors["true_sentiment"] = ""

# Keep only necessary columns
error_analysis = sentiment_errors[
    [
        "review_id",
        "sentence",
        "aspect",
        "aspect_sentiment",
        "aspect_score",
        "true_sentiment"
    ]
]

# Save
error_analysis.to_csv(
    "/kaggle/working/sentiment_error_analysis.csv",
    index=False
)

print(error_analysis.shape)

(55, 6)


In [29]:
errors = pd.read_excel("/kaggle/input/datasets/kylin0338/error-finding/sentiment_error_analysis.csv.xlsx"
)

In [30]:
from sklearn.metrics import confusion_matrix, classification_report

print(classification_report(
    errors["true_sentiment"],
    errors["aspect_sentiment"],
    labels=["positive", "negative", "neutral"]
))

              precision    recall  f1-score   support

    positive       0.18      0.11      0.14        18
    negative       0.47      0.30      0.36        27
     neutral       0.04      0.20      0.06         5

   micro avg       0.20      0.22      0.21        50
   macro avg       0.23      0.20      0.19        50
weighted avg       0.32      0.22      0.25        50



In [31]:
print(confusion_matrix(
    errors["true_sentiment"],
    errors["aspect_sentiment"],
    labels=["positive", "negative", "neutral"]
))

[[ 2  3 13]
 [ 7  8 12]
 [ 1  3  1]]


In [32]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/__results__.html
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/__huggingface_repos__.json
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/__notebook__.ipynb
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/reviews_with_sentiment.csv
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/__output__.json
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/custom.css
/kaggle/input/datasets/kylin0338/pairswithprobs/pairs_with_probs.csv
/kaggle/input/datasets/kylin0338/error-finding/sentiment_error_analysis.csv.xlsx
/kaggle/input/datasets/kylin0338/updated-validation/validation.csv.xlsx
/kaggle/input/datasets/charitarth/semeval-2014-task-4-aspectbasedsentimentanalysis/Laptop_Train_v2.xml
/kaggle/input/datasets/charitarth/semeval-2014-task-4-aspectbasedsentimentanalysis/Restaurants_Test_Data_PhaseB.csv
/kaggle/input/datasets/charitarth/semeval-2014-task-4-aspectbasedsentimentanalysis/laptops-trial.csv
/kagg

In [33]:
v = pd.read_excel("/kaggle/input/datasets/kylin0338/updated-validation/validation.csv.xlsx")

errors = pd.read_excel("/kaggle/input/datasets/kylin0338/error-finding/sentiment_error_analysis.csv.xlsx")

In [34]:
print("Validation columns:")
print(v.columns.tolist())

print("\nError analysis columns:")
print(errors.columns.tolist())

print("\nValidation shape:", v.shape)
print("Error analysis shape:", errors.shape)

print("\nMissing true_sentiment labels:")
print(errors["true_sentiment"].isna().sum())

Validation columns:
['review_id', 'sentence', 'aspect', 'aspect_sentiment', 'aspect_score', 'aspect_correct', 'sentiment_correct']

Error analysis columns:
['review_id', 'sentence', 'aspect', 'aspect_sentiment', 'aspect_score', 'true_sentiment']

Validation shape: (280, 7)
Error analysis shape: (55, 6)

Missing true_sentiment labels:
0


In [35]:
print("Validation:")
print(v[["review_id", "aspect"]].head())

print("\nErrors:")
print(errors[["review_id", "aspect", "true_sentiment"]].head())

Validation:
   review_id   aspect
0       6719  battery
1      15636  battery
2       7156  battery
3       7638  battery
4      17304  battery

Errors:
   review_id   aspect true_sentiment
0       7156  battery       positive
1       7638  battery       negative
2      17304  battery       negative
3       3657  battery        neutral
4       3821  battery       negative


In [36]:
# Columns used to match the rows
keys = ["review_id", "sentence", "aspect"]

# Keep only the matching columns and corrected labels
corrected = errors[keys + ["true_sentiment"]].copy()

# Check that the error file has no duplicate matching rows
print("Duplicate error rows:", corrected.duplicated(keys).sum())

# Merge corrected labels into validation
v = v.merge(
    corrected,
    on=keys,
    how="left",
    validate="many_to_one"
)

# For correctly predicted sentiment rows, predicted = true
v.loc[
    v["sentiment_correct"] == 1,
    "true_sentiment"
] = v.loc[
    v["sentiment_correct"] == 1,
    "aspect_sentiment"
]

print("Updated shape:", v.shape)
print("Missing true labels:", v["true_sentiment"].isna().sum())

print(v[["aspect", "aspect_sentiment", "true_sentiment"]].head(10))

Duplicate error rows: 0
Updated shape: (280, 8)
Missing true labels: 13
    aspect aspect_sentiment true_sentiment
0  battery         negative       negative
1  battery         negative       negative
2  battery          neutral       positive
3  battery         positive       negative
4  battery         negative       negative
5  battery          neutral            NaN
6  battery         negative       negative
7  battery         negative       negative
8  battery         negative       negative
9  battery         negative       negative


In [37]:
# Check the validation flags for rows with missing true sentiment
missing = v[v["true_sentiment"].isna()]

print("Missing rows:", len(missing))

print("\nAspect correctness:")
print(missing["aspect_correct"].value_counts(dropna=False))

print("\nSentiment correctness:")
print(missing["sentiment_correct"].value_counts(dropna=False))

print("\nMissing rows:")
display(missing[
    ["review_id", "sentence", "aspect",
     "aspect_sentiment", "aspect_correct", "sentiment_correct"]
])

Missing rows: 13

Aspect correctness:
aspect_correct
0    11
1     2
Name: count, dtype: int64

Sentiment correctness:
sentiment_correct
0    13
Name: count, dtype: int64

Missing rows:


,review_id,sentence,aspect,aspect_sentiment,aspect_correct,sentiment_correct
5,5016,"When I plugged the phone in to charge and turned it on and connected it to my Wi-Fi and started downloading its update information, Google Drive kept crashing.",battery,neutral,0,0
20,16208,"Thereâ€™s no big scratch, charging port is little bit worn.",build,positive,1,0
54,3756,"The phone is good but the battery heats up when taking a video, using the camera, GPS or playing a video for just under 5 min.",camera,neutral,0,0
80,11538,And from the day out of the box it was locking up terrible.,delivery,neutral,0,0
96,18175,"As soon as I opened the box, I could tell right away the 2XL wasn't going to work out but I thought maybe I would get used to it.",delivery,neutral,0,0
101,13296,Got a slim case with it but am using the case I ordered separately with tempered glass screen protector.,design,positive,0,0
107,2017,"Looks great, fingerprint works but not quick fast in response.",design,positive,0,0
113,13339,"What moron creates a device that films in briliant colors, and details, but limits the user to a 14 min of storage.",design,positive,0,0
161,15071,Other phones in this price range can't even touch the quality of the camera for pictures.,price,neutral,0,0
178,15245,Itâ€™s also in very good condition based on the price.,price,neutral,1,0


In [38]:
check = v[
    v["review_id"].isin([16208, 15245])
][["review_id", "sentence", "aspect"]]

print("Rows in validation:")
display(check)

print("Matching rows in error file:")
display(
    errors[
        errors["review_id"].isin([16208, 15245])
    ][["review_id", "sentence", "aspect", "true_sentiment"]]
)

Rows in validation:


,review_id,sentence,aspect
20,16208,"Thereâ€™s no big scratch, charging port is little bit worn.",build
178,15245,Itâ€™s also in very good condition based on the price.,price


Matching rows in error file:


,review_id,sentence,aspect,true_sentiment
5,16208,"ThereÃ¢â‚¬â„¢s no big scratch, charging port is little bit worn.",build,neutral
38,15245,ItÃ¢â‚¬â„¢s also in very good condition based on the price.,price,positive


In [39]:
# Create a lookup using review_id and aspect
corrected_lookup = errors.set_index(
    ["review_id", "aspect"]
)["true_sentiment"]

# Fill missing labels using the matching review_id and aspect
for idx in v[v["true_sentiment"].isna()].index:
    key = (v.loc[idx, "review_id"], v.loc[idx, "aspect"])

    if key in corrected_lookup.index:
        v.loc[idx, "true_sentiment"] = corrected_lookup.loc[key]

print("Missing labels after correction:",
      v["true_sentiment"].isna().sum())

print("Updated shape:", v.shape)

Missing labels after correction: 11
Updated shape: (280, 8)


In [40]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Exclude incorrectly detected aspects
sentiment_eval = v[v["aspect_correct"] == 1].copy()

y_true = sentiment_eval["true_sentiment"]
y_pred = sentiment_eval["aspect_sentiment"]

labels = ["negative", "neutral", "positive"]

print("Evaluation samples:", len(sentiment_eval))

print("\nAccuracy:")
print(accuracy_score(y_true, y_pred))

print("\nMacro F1:")
print(f1_score(y_true, y_pred, labels=labels, average="macro"))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=labels))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=labels
))

Evaluation samples: 253

Accuracy:
0.8260869565217391

Macro F1:
0.7849620279960837

Confusion Matrix:
[[88 12  7]
 [ 3 23  1]
 [ 3 13 98]]

Classification Report:
              precision    recall  f1-score   support

    negative       0.91      0.82      0.86       107
     neutral       0.47      0.85      0.61        27
    positive       0.92      0.86      0.89       114

   micro avg       0.83      0.84      0.83       248
   macro avg       0.76      0.84      0.78       248
weighted avg       0.86      0.84      0.85       248



In [41]:
print("True sentiment values:")
print(y_true.value_counts(dropna=False))

print("\nPredicted sentiment values:")
print(y_pred.value_counts(dropna=False))

print("\nUnexpected true labels:")
print(y_true[~y_true.isin(labels)].unique())

print("\nUnexpected predicted labels:")
print(y_pred[~y_pred.isin(labels)].unique())

True sentiment values:
true_sentiment
positive    114
negative    107
neutral      27
neutal        5
Name: count, dtype: int64

Predicted sentiment values:
aspect_sentiment
positive    107
negative     97
neutral      49
Name: count, dtype: int64

Unexpected true labels:
['neutal']

Unexpected predicted labels:
[]


In [42]:
v["true_sentiment"] = v["true_sentiment"].replace({
    "neutal": "neutral"
})

# Recreate evaluation dataset
sentiment_eval = v[v["aspect_correct"] == 1].copy()

y_true = sentiment_eval["true_sentiment"]
y_pred = sentiment_eval["aspect_sentiment"]

print(y_true.value_counts())
print("\nTotal samples:", len(sentiment_eval))

true_sentiment
positive    114
negative    107
neutral      32
Name: count, dtype: int64

Total samples: 253


In [43]:
print("Accuracy:", accuracy_score(y_true, y_pred))

print("Macro F1:", f1_score(
    y_true, y_pred, labels=labels, average="macro"
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=labels))

print("\nClassification Report:")
print(classification_report(
    y_true, y_pred, labels=labels
))

Accuracy: 0.8300395256916996
Macro F1: 0.7807385062287023

Confusion Matrix:
[[88 12  7]
 [ 6 24  2]
 [ 3 13 98]]

Classification Report:
              precision    recall  f1-score   support

    negative       0.91      0.82      0.86       107
     neutral       0.49      0.75      0.59        32
    positive       0.92      0.86      0.89       114

    accuracy                           0.83       253
   macro avg       0.77      0.81      0.78       253
weighted avg       0.86      0.83      0.84       253



In [44]:
neutral_errors = sentiment_eval[
    (sentiment_eval["aspect_sentiment"] == "neutral") &
    (sentiment_eval["true_sentiment"] != "neutral")
]

print("Neutral misclassifications:", len(neutral_errors))

display(neutral_errors[
    ["review_id", "sentence", "aspect",
     "aspect_sentiment", "true_sentiment", "aspect_score"]
])

Neutral misclassifications: 25


,review_id,sentence,aspect,aspect_sentiment,true_sentiment,aspect_score
2,7156,"However, now, I charge it every other night with normal use.",battery,neutral,positive,0.610037
17,3821,I have to dock the battery life a star because of its small size.,battery,neutral,negative,0.696890
32,3096,"The phone is not fragile, many nights I lay on top of it while it is charging.",build,neutral,positive,0.955093
33,8067,Build Quality - Not too bad but the Asus is way better with its premium metal feel.,build,neutral,negative,0.730827
41,6943,Now when you see a photo you want to take you don't have to fumble for the camera app.,camera,neutral,positive,0.650090
49,9432,Patience is going to be the key while dealing with the camera app on this phone.,camera,neutral,negative,0.744272
52,6260,"The physical button just works faster for me and it also seems to be easier to explain ""press here"" to some stranger who I ask to take a photo of me with my camera.",camera,neutral,positive,0.997111
74,2274,"If I ever need another phone, I will most likely purchase from this seller.",customer_service,neutral,positive,0.857598
94,16580,"The turbo charger that came with the phone did not work, right out of the box.",delivery,neutral,negative,0.545371
122,9563,Took my sim card out of ATT and put it in my new phone and works great.,network,neutral,positive,0.668174


In [45]:
print(absa.model.config.id2label)

{0: 'Negative', 1: 'Neutral', 2: 'Positive'}


In [46]:
out_all = absa(
    inputs[:5],
    batch_size=5,
    top_k=None
)

print(out_all)

[[{'label': 'Negative', 'score': 0.997238278388977}, {'label': 'Neutral', 'score': 0.001654102117754519}, {'label': 'Positive', 'score': 0.0011076262453570962}], [{'label': 'Negative', 'score': 0.9944666028022766}, {'label': 'Neutral', 'score': 0.004896821919828653}, {'label': 'Positive', 'score': 0.0006365959998220205}], [{'label': 'Negative', 'score': 0.996756374835968}, {'label': 'Neutral', 'score': 0.0026115612126886845}, {'label': 'Positive', 'score': 0.0006320292595773935}], [{'label': 'Negative', 'score': 0.9966999888420105}, {'label': 'Neutral', 'score': 0.0023450141306966543}, {'label': 'Positive', 'score': 0.0009549116366542876}], [{'label': 'Positive', 'score': 0.9957242012023926}, {'label': 'Neutral', 'score': 0.0034921015612781048}, {'label': 'Negative', 'score': 0.0007837468874640763}]]


In [47]:
# Prepare sentence-aspect pairs from the validation dataset
eval_inputs = [
    {"text": sentence, "text_pair": aspect}
    for sentence, aspect in zip(
        sentiment_eval["sentence"],
        sentiment_eval["aspect"]
    )
]

# Get all three sentiment probabilities
eval_out = absa(
    eval_inputs,
    batch_size=64,
    top_k=None
)

# Convert probabilities into separate columns
probabilities = pd.DataFrame([
    {
        item["label"].lower(): item["score"]
        for item in result
    }
    for result in eval_out
])

# Attach probabilities to validation data
sentiment_eval = sentiment_eval.reset_index(drop=True)
probabilities = probabilities.reset_index(drop=True)

sentiment_eval = pd.concat(
    [sentiment_eval, probabilities],
    axis=1
)

print(sentiment_eval[
    ["aspect", "aspect_sentiment", "true_sentiment",
     "negative", "neutral", "positive"]
].head())

print("\nTotal samples:", len(sentiment_eval))

    aspect aspect_sentiment true_sentiment  negative   neutral  positive
0  battery         negative       negative  0.990458  0.005203  0.004339
1  battery          neutral       positive  0.374296  0.610039  0.015665
2  battery         positive       negative  0.001366  0.002851  0.995783
3  battery         negative       negative  0.936439  0.058338  0.005223
4  battery         negative       negative  0.996558  0.001040  0.002402

Total samples: 253


In [48]:
neutral_errors = sentiment_eval[
    (sentiment_eval["aspect_sentiment"] == "neutral") &
    (sentiment_eval["true_sentiment"] != "neutral")
].copy()

display(
    neutral_errors[
        ["review_id", "aspect", "true_sentiment",
         "negative", "neutral", "positive"]
    ].sort_values("neutral", ascending=False)
)

,review_id,aspect,true_sentiment,negative,neutral,positive
50,6260,camera,positive,0.000864,0.997111,0.002025
233,19321,storage,negative,0.008314,0.990079,0.001607
160,15245,price,positive,0.007165,0.967006,0.025829
30,3096,build,positive,0.010200,0.955093,0.034707
186,16605,software,positive,0.005643,0.915093,0.079263
206,3932,sound,negative,0.053431,0.892609,0.053960
187,3096,software,negative,0.060923,0.885996,0.053081
70,2274,customer_service,positive,0.033275,0.857598,0.109127
174,10232,screen,positive,0.007768,0.761132,0.231100
47,9432,camera,negative,0.232527,0.744273,0.023199


In [49]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Split the validation dataset
tune, test = train_test_split(
    sentiment_eval,
    test_size=0.4,
    random_state=42,
    stratify=sentiment_eval["true_sentiment"]
)

# Function to apply neutral threshold
def predict_threshold(data, threshold):

    predictions = []

    for _, row in data.iterrows():

        if row["neutral"] >= threshold:
            predictions.append("neutral")

        elif row["positive"] > row["negative"]:
            predictions.append("positive")

        else:
            predictions.append("negative")

    return predictions


# Test different thresholds
thresholds = np.arange(0.50, 1.00, 0.05)

results = []

for threshold in thresholds:

    predictions = predict_threshold(tune, threshold)

    accuracy = accuracy_score(
        tune["true_sentiment"], predictions
    )

    macro_f1 = f1_score(
        tune["true_sentiment"],
        predictions,
        labels=labels,
        average="macro"
    )

    results.append({
        "threshold": round(threshold, 2),
        "accuracy": accuracy,
        "macro_f1": macro_f1
    })

results_df = pd.DataFrame(results)

display(results_df.sort_values("macro_f1", ascending=False))

,threshold,accuracy,macro_f1
4,0.70,0.880795,0.824814
3,0.65,0.854305,0.798345
2,0.60,0.847682,0.791027
1,0.55,0.847682,0.791027
5,0.75,0.867550,0.789620
0,0.50,0.841060,0.783650
6,0.80,0.860927,0.771970
7,0.85,0.854305,0.753321
8,0.90,0.860927,0.752616
9,0.95,0.841060,0.679500


In [50]:
best_threshold = results_df.loc[
    results_df["macro_f1"].idxmax(),
    "threshold"
]

print("Selected threshold:", best_threshold)

# Evaluate on unseen test portion
test_predictions = predict_threshold(test, best_threshold)

print("\nAccuracy:")
print(accuracy_score(test["true_sentiment"], test_predictions))

print("\nMacro F1:")
print(f1_score(
    test["true_sentiment"],
    test_predictions,
    labels=labels,
    average="macro"
))

print("\nClassification Report:")
print(classification_report(
    test["true_sentiment"],
    test_predictions,
    labels=labels
))

Selected threshold: 0.7

Accuracy:
0.7941176470588235

Macro F1:
0.7186913086913087

Classification Report:
              precision    recall  f1-score   support

    negative       0.82      0.86      0.84        43
     neutral       0.50      0.46      0.48        13
    positive       0.84      0.83      0.84        46

    accuracy                           0.79       102
   macro avg       0.72      0.72      0.72       102
weighted avg       0.79      0.79      0.79       102



In [51]:
print("Original model macro F1:")

print(f1_score(
    test["true_sentiment"],
    test["aspect_sentiment"],
    labels=labels,
    average="macro"
))

Original model macro F1:
0.7608932461873638


In [52]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/__results__.html
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/__huggingface_repos__.json
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/__notebook__.ipynb
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/reviews_with_sentiment.csv
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/__output__.json
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/custom.css
/kaggle/input/datasets/kylin0338/pairswithprobs/pairs_with_probs.csv
/kaggle/input/datasets/kylin0338/error-finding/sentiment_error_analysis.csv.xlsx
/kaggle/input/datasets/kylin0338/updated-validation/validation.csv.xlsx
/kaggle/input/datasets/charitarth/semeval-2014-task-4-aspectbasedsentimentanalysis/Laptop_Train_v2.xml
/kaggle/input/datasets/charitarth/semeval-2014-task-4-aspectbasedsentimentanalysis/Restaurants_Test_Data_PhaseB.csv
/kaggle/input/datasets/charitarth/semeval-2014-task-4-aspectbasedsentimentanalysis/laptops-trial.csv
/kagg

In [53]:
import xml.etree.ElementTree as ET

base = "/kaggle/input/datasets/charitarth/semeval-2014-task-4-aspectbasedsentimentanalysis/"

train_path = base + "Laptop_Train_v2.xml"
test_path = base + "Laptops_Test_Data_phaseB.xml"

train_root = ET.parse(train_path).getroot()
test_root = ET.parse(test_path).getroot()

print("Training root:", train_root.tag)
print("Testing root:", test_root.tag)

print("\nTRAIN SAMPLE:")
print(ET.tostring(train_root[0], encoding="unicode")[:2500])

print("\nTEST SAMPLE:")
print(ET.tostring(test_root[0], encoding="unicode")[:2500])

Training root: sentences
Testing root: sentences

TRAIN SAMPLE:
<sentence id="2339">
        <text>I charge it at night and skip taking the cord with me because of the good battery life.</text>
        <aspectTerms>
            <aspectTerm term="cord" polarity="neutral" from="41" to="45" />
            <aspectTerm term="battery life" polarity="positive" from="74" to="86" />
        </aspectTerms>
    </sentence>
    

TEST SAMPLE:
<sentence id="892:1">
        <text>Boot time is super fast, around anywhere from 35 seconds to 1 minute.</text>
        <aspectTerms>
            <aspectTerm term="Boot time" from="0" to="9" />
        </aspectTerms>
    </sentence>
    


In [54]:
import pandas as pd

train_csv = pd.read_csv(base + "Laptop_Train_v2.csv")
test_csv = pd.read_csv(base + "Laptops_Test_Data_PhaseB.csv")

print("Training shape:", train_csv.shape)
print("Training columns:", train_csv.columns.tolist())

print("\nTest shape:", test_csv.shape)
print("Test columns:", test_csv.columns.tolist())

display(train_csv.head(5))
display(test_csv.head(5))

Training shape: (2358, 6)
Training columns: ['id', 'Sentence', 'Aspect Term', 'polarity', 'from', 'to']

Test shape: (800, 2)
Test columns: ['id', 'Sentence']


,id,Sentence,Aspect Term,polarity,from,to
0,2339,I charge it at night and skip taking the cord with me because of the good battery life.,cord,neutral,41,45
1,2339,I charge it at night and skip taking the cord with me because of the good battery life.,battery life,positive,74,86
2,1316,"The tech guy then said the service center does not do 1-to-1 exchange and I have to direct my concern to the ""sales"" team, which is the retail shop which I bought my netbook from.",service center,negative,27,41
3,1316,"The tech guy then said the service center does not do 1-to-1 exchange and I have to direct my concern to the ""sales"" team, which is the retail shop which I bought my netbook from.","""sales"" team",negative,109,121
4,1316,"The tech guy then said the service center does not do 1-to-1 exchange and I have to direct my concern to the ""sales"" team, which is the retail shop which I bought my netbook from.",tech guy,neutral,4,12


,id,Sentence
0,892:1,"Boot time is super fast, around anywhere from 35 seconds to 1 minute."
1,1144:1,tech support would not fix the problem unless I bought your plan for $150 plus.
2,805:2,but in resume this computer rocks!
3,359:1,Set up was easy.
4,562:1,Did not enjoy the new Windows 8 and touchscreen functions.


In [55]:
print("Training polarity distribution:")
print(train_csv["polarity"].value_counts())

print("\nMissing values:")
print(train_csv.isnull().sum())

print("\nUnique polarity labels:")
print(train_csv["polarity"].unique())

print("\nTraining examples:")
display(train_csv.sample(10, random_state=42))

Training polarity distribution:
polarity
positive    987
negative    866
neutral     460
conflict     45
Name: count, dtype: int64

Missing values:
id             0
Sentence       0
Aspect Term    0
polarity       0
from           0
to             0
dtype: int64

Unique polarity labels:
['neutral' 'positive' 'negative' 'conflict']

Training examples:


,id,Sentence,Aspect Term,polarity,from,to
342,2496,Mac is not made for gaming.,gaming,negative,20,26
1137,1494,"I made a photo book as a gift, on my computer, pushed ""Buy"" and it drew from my iTunes account and sent the book to my house, the book was of perfect quality-exactly how I had created it and looked better than I had even imagined.",iTunes,neutral,80,86
678,2947,wonderful features.,features,positive,10,18
361,1068,"I reloaded with Windows 7 Ultimate, and the Bluetooth and Fingerprint reader (software) would not load.",Fingerprint reader,negative,58,76
1602,2917,The notebook is lacking a HDMI port and a S-video port that would enable one to hook it to a TV.,S-video port,negative,42,54
2112,2722,I'm really impressed with the quality and performance for the price of the computer.,performance,positive,42,53
1128,1330,This is what I call a good after sales service.,after sales service,positive,27,46
247,2708,The start menu is not the easiest thing to navigate due to the stacking.,navigate,negative,43,51
464,1623,With a MAC computer I have more free time as I don't have to wait for windows to boot up or shut down and all the viruses associated with windows.,boot up,negative,81,88
1656,2236,"also you may need to charge it once a day, if for medium use every thing fast and easy with mac the size and look is the most feature that attracted me to it.",charge,neutral,21,27


In [56]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Keep only the three classes supported by DeBERTa
semeval = train_csv[
    train_csv["polarity"].isin(["positive", "negative", "neutral"])
].copy()

# Keep only the columns needed for ABSA
semeval = semeval[
    ["id", "Sentence", "Aspect Term", "polarity"]
].copy()

# Rename columns to match our model input format
semeval = semeval.rename(columns={
    "id": "sentence_id",
    "Sentence": "sentence",
    "Aspect Term": "aspect",
    "polarity": "true_sentiment"
})

# Group split by sentence ID
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(semeval, groups=semeval["sentence_id"])
)

semeval_train = semeval.iloc[train_idx].copy()
semeval_test = semeval.iloc[test_idx].copy()

print("Total usable samples:", len(semeval))
print("Training samples:", len(semeval_train))
print("Evaluation samples:", len(semeval_test))

print("\nEvaluation sentiment distribution:")
print(semeval_test["true_sentiment"].value_counts())

print("\nEvaluation sample:")
display(semeval_test.head())

Total usable samples: 2313
Training samples: 1878
Evaluation samples: 435

Evaluation sentiment distribution:
true_sentiment
positive    192
negative    162
neutral      81
Name: count, dtype: int64

Evaluation sample:


,sentence_id,sentence,aspect,true_sentiment
0,2339,I charge it at night and skip taking the cord with me because of the good battery life.,cord,neutral
1,2339,I charge it at night and skip taking the cord with me because of the good battery life.,battery life,positive
9,2789,Easy to start up and does not overheat as much as other laptops.,start up,positive
26,2568,I love the way the entire suite of software works together.,suite of software,positive
27,295,The speed is incredible and I am more than satisfied.,speed,positive


In [57]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# Prepare sentence-aspect pairs
inputs = [
    {
        "text": sentence,
        "text_pair": aspect
    }
    for sentence, aspect in zip(
        semeval_test["sentence"],
        semeval_test["aspect"]
    )
]

# Predict using your existing DeBERTa model
predictions = absa(
    inputs,
    batch_size=64
)

# Store predictions
semeval_test["predicted_sentiment"] = [
    result["label"].lower()
    for result in predictions
]

semeval_test["prediction_score"] = [
    result["score"]
    for result in predictions
]

# Calculate metrics
labels = ["negative", "neutral", "positive"]

y_true = semeval_test["true_sentiment"]
y_pred = semeval_test["predicted_sentiment"]

print("Accuracy:", accuracy_score(y_true, y_pred))

print(
    "Macro F1:",
    f1_score(y_true, y_pred, labels=labels, average="macro")
)

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=labels,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=labels))

Accuracy: 0.9540229885057471
Macro F1: 0.9424016363296256

Classification Report:
              precision    recall  f1-score   support

    negative       0.97      0.98      0.98       162
     neutral       0.86      0.91      0.89        81
    positive       0.98      0.95      0.97       192

    accuracy                           0.95       435
   macro avg       0.94      0.95      0.94       435
weighted avg       0.96      0.95      0.95       435


Confusion Matrix:
[[159   2   1]
 [  5  74   2]
 [  0  10 182]]
